# Topic-Based Video Clip Agent

An agent that takes a video file and a list of topics/keywords, and
produces one trimmed `.mp4` clip per topic — cut on a natural pause rather
than a hard timestamp — while guaranteeing that running it twice on the
same input never creates duplicate or partially-overwritten files.

## Pipeline

```
video -> speech-to-text (word/sentence-level timestamps)
      -> topic chaptering (decide the full time range each topic covers)
      -> acoustic silence detection (independent ffmpeg pass)
      -> snap each chapter's start/end to the nearest real silence
      -> ffmpeg extraction (atomic write)
      -> persistent JSON state (atomic write, powers idempotency)
```

## Design notes

**Chaptering over chunking.** Rather than slicing the transcript into many
small natural-pause chunks and picking the single best-scoring chunk per
topic (which tends to produce a short, arbitrary-looking slice of
whoever's talking *near* the keyword match), this agent first decides the
**full contiguous time range** each topic covers across the whole
transcript, and only then snaps just the two edges of that range to a
nearby silence/pause. That way a clip covers the topic's entire discussion,
not just the sentence where the keyword happened to appear.

**Chaptering method**: a rule-based, monotonic dynamic-programming keyword
matcher — token-overlap scoring per sentence, resolved into non-overlapping
chapters under the constraint that the topic index can only stay the same
or advance by one as time moves forward (topics are assumed to be
discussed once each, in the order given). This removes flip-flopping caused
by incidental shared vocabulary between topics (see Section 8 for a
concrete example and its one known limitation).

**Idempotency**: content-addressable clip IDs, atomic file/state writes,
drift detection between state and disk (`reconcile_state()`), and a
post-hoc boundary check (`verify_clean_boundaries()`) — details in
Section 4.

## Setup (run once, locally)
```bash
pip install -r requirements.txt
# ffmpeg must be on PATH (provides both `ffmpeg` and `ffprobe`)
```


In [ ]:
import os, json, hashlib, subprocess, time, re
from pathlib import Path
from typing import List, Dict, Optional, Tuple

VIDEO_DIR = Path("videos")
OUTPUT_DIR = Path("clips")
STATE_FILE = Path("clip_state.json")
TRANSCRIPT_CACHE_DIR = Path("transcript_cache")

for d in (VIDEO_DIR, OUTPUT_DIR, TRANSCRIPT_CACHE_DIR):
    d.mkdir(exist_ok=True)

def sha256_file(path, chunk=1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


## 1. Speech-to-text with word- and sentence-level timestamps

`faster-whisper` gives per-word and per-sentence start/end times. Transcripts
are cached on disk keyed by the video file's sha256, so re-running on an
unchanged file never re-transcribes — the expensive step, and the first
idempotency win.

Note: the `faster_whisper` import is **lazy** (only happens on a cache miss).
That way a cached re-run works even in an environment where the package
isn't installed, instead of failing on an unconditional top-level import.


In [ ]:
def get_model():
    from faster_whisper import WhisperModel
    return WhisperModel("small", device="cpu", compute_type="int8")

def transcribe(video_path: Path) -> dict:
    video_hash = sha256_file(video_path)
    cache_path = TRANSCRIPT_CACHE_DIR / f"{video_hash}.json"
    if cache_path.exists():
        print(f"[cache] transcript already exists for {video_path.name} -> reusing, no re-transcription")
        return json.loads(cache_path.read_text())

    print(f"[whisper] transcribing {video_path.name} ...")
    model = get_model()
    segments, info = model.transcribe(str(video_path), word_timestamps=True)

    words, seg_list = [], []
    for seg in segments:
        seg_list.append({"start": seg.start, "end": seg.end, "text": seg.text.strip()})
        for w in (seg.words or []):
            if w.word.strip():
                words.append({"word": w.word.strip(), "start": w.start, "end": w.end})

    result = {
        "video_hash": video_hash,
        "video_path": str(video_path),
        "language": info.language,
        "words": words,
        "segments": seg_list,
    }
    cache_path.write_text(json.dumps(result, indent=2))
    return result


## 2. Acoustic silence detection (ground truth for boundary snapping)

Independent of Whisper: `ffmpeg silencedetect` finds real silence intervals
in the source audio by measuring actual signal level, not by inferring
pauses from the transcript. This is used later in the pipeline to snap a
topic's (coarse) start/end time onto the nearest real pause, so the final
cut doesn't land mid-word.


In [ ]:
def find_silences(video_path, noise_db=-30, min_dur=0.08) -> List[Tuple[float, float]]:
    """Run ffmpeg's silencedetect filter and return (silence_start, silence_end) pairs in seconds."""
    cmd = ["ffmpeg", "-i", str(video_path), "-af",
           f"silencedetect=noise={noise_db}dB:d={min_dur}", "-f", "null", "-"]
    log = subprocess.run(cmd, capture_output=True, text=True).stderr
    starts = [float(x) for x in re.findall(r"silence_start:\s*(-?[\d.]+)", log)]
    ends   = [float(x) for x in re.findall(r"silence_end:\s*(-?[\d.]+)", log)]
    return list(zip(starts, ends))

def snap_chapter_boundary(t: float, silences: List[Tuple[float, float]], window: float = 6.0) -> float:
    """
    Snap a coarse chapter boundary to the MIDPOINT of the nearest real silence
    interval within `window` seconds. Chapter boundaries come from sentence-
    level timestamps (already natural sentence breaks), so they're usually
    close to a real pause already; this snap removes the last bit of drift so
    the cut lands cleanly on silence rather than a fraction of a second into
    the next/previous word. `window` is deliberately generous (6s) because a
    *chapter* boundary can be off by a couple of seconds even when it's
    correctly chosen -- sentence timestamps aren't perfectly tight to the
    pause the way a single word's is.
    """
    best, best_d = None, window + 1e-9
    for s, e in silences:
        mid = (s + e) / 2
        d = abs(mid - t)
        if d <= window and d < best_d:
            best, best_d = mid, d
    return best if best is not None else t


## 3. Topic chaptering: decide the FULL span each topic covers

This is the core decision-making step: rather than matching a topic to a
single sentence, it decides the full contiguous time range each topic
covers across the *whole* transcript, and only then hands the two edges of
that range to the silence-snap step (Section 5).

**Method**: token-overlap keyword scoring per sentence, resolved with a
**monotonic DP**: each sentence is assigned to whichever topic scores
highest, under the constraint that the topic index can only stay the same
or advance by one as time moves forward (topics are assumed to occur once
each, in the order given). Enforcing monotonicity is what prevents
flip-flopping between topics that happen to share vocabulary (see Section 8
for a concrete example on the sample video, where "database" and "table"
show up in more than one section). Each topic's chaptering confidence is
also logged, so a weak match is visible rather than silently trusted.

*With more time*, this scoring function is the natural place to swap in a
single LLM call over the indexed/timestamped transcript instead of keyword
matching — see Section 8 and "Next steps" at the end for why, and what
that would fix.


In [ ]:
import difflib

STOPWORDS = {"a","an","the","and","or","of","to","in","on","for","with","this","that",
             "is","are","we","our","its","it's","as","by","from","which","was","were"}

def _tokenize(s: str) -> List[str]:
    return [t for t in re.findall(r"[a-z0-9']+", s.lower()) if t not in STOPWORDS]

def keyword_score(text: str, topic: str) -> float:
    """Token-overlap score (fraction of the topic's words found in the text),
    plus a bonus if the topic appears as a literal substring. Scoring by
    token overlap, rather than a character-level similarity ratio over the
    WHOLE sentence, avoids noisy false-positive matches between a short
    topic phrase and a long, unrelated sentence that happens to share some
    letters."""
    text_tokens = set(_tokenize(text))
    topic_tokens = _tokenize(topic)
    if not topic_tokens:
        return 0.0
    hits = sum(1 for t in topic_tokens if t in text_tokens)
    coverage = hits / len(topic_tokens)
    phrase_bonus = 1.0 if topic.lower() in text.lower() else 0.0
    return coverage + phrase_bonus


def rule_based_chapter_topics(segments: List[Dict], topics: List[str]):
    """Monotonic-DP keyword chaptering: assigns each transcript sentence to
    a topic and resolves the assignment into non-overlapping chapters.
    Returns (spans, confidence) where confidence is each topic's average
    per-sentence keyword_score over its assigned span -- a low value is a
    signal the match is weak and the caller should warn rather than
    silently trust it."""
    n, K = len(segments), len(topics)
    emission = [[keyword_score(seg["text"], t) for t in topics] for seg in segments]
    NEG = float("-inf")
    dp = [[NEG] * K for _ in range(n)]
    back = [[None] * K for _ in range(n)]
    dp[0][0] = emission[0][0]
    for i in range(1, n):
        for j in range(K):
            best_prev, best_val = None, NEG
            if dp[i - 1][j] > best_val:
                best_val, best_prev = dp[i - 1][j], j
            if j > 0 and dp[i - 1][j - 1] > best_val:
                best_val, best_prev = dp[i - 1][j - 1], j - 1
            if best_prev is not None:
                dp[i][j] = best_val + emission[i][j]
                back[i][j] = best_prev
    end_j = max(range(K), key=lambda j: dp[n - 1][j])
    labels = [None] * n
    labels[n - 1] = end_j
    for i in range(n - 1, 0, -1):
        labels[i - 1] = back[i][labels[i]]

    spans, confidence = {}, {}
    i = 0
    while i < n:
        j = labels[i]
        start_i = i
        while i < n and labels[i] == j:
            i += 1
        end_i = i - 1
        topic = topics[j]
        span = (segments[start_i]["start"], segments[end_i]["end"])
        if topic not in spans or (span[1] - span[0]) > (spans[topic][1] - spans[topic][0]):
            spans[topic] = span
            confidence[topic] = sum(emission[k][j] for k in range(start_i, end_i + 1)) / (end_i - start_i + 1)
    return spans, confidence


def chapter_topics(segments: List[Dict], topics: List[str]) -> Dict[str, Tuple[float, float]]:
    """Entry point for the chaptering step. Kept as a single function so the
    orchestrator (Section 6) doesn't need to know which chaptering method is
    used underneath -- swapping in an LLM-based chaptering method later
    would only mean changing what happens inside this function."""
    print("[chapter] using monotonic keyword chaptering")
    spans, confidence = rule_based_chapter_topics(segments, topics)
    for t, c in confidence.items():
        if c < 0.3:
            print(f"[warn] low keyword-confidence ({c:.2f}) chaptering topic {t!r} -- "
                  f"-- topic wording may differ from transcript wording")
    return spans


## 4. Persistent state + idempotency

This is what makes re-running the agent on the same input safe:
- Clip ID = `sha1(video_hash : topic : raw_start : raw_end)`, computed from
  the **pre-snap** chapter boundaries, so identity is stable even if
  silence-snap parameters are retuned later.
- State writes are atomic (`.tmp` + `os.replace`) so a crash mid-write
  can't corrupt `clip_state.json`.
- Clip files are written to `.part` then atomically renamed, so a crash
  mid-encode can't leave a partial `.mp4` behind.
- Before skipping a "known" clip, the agent checks the file **actually
  exists on disk** — state saying "done" is never trusted blindly.
- `reconcile_state()` drops any state entry whose file is missing, and
  reports (without deleting) any `.mp4` on disk that isn't tracked by
  state, so drift between state and disk is visible rather than silent.


In [ ]:
def load_state() -> dict:
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {}

def save_state(state: dict):
    tmp = STATE_FILE.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=2, sort_keys=True))
    os.replace(tmp, STATE_FILE)

def make_clip_id(video_hash: str, topic: str, start: float, end: float) -> str:
    key = f"{video_hash}:{topic.lower().strip()}:{round(start, 2)}:{round(end, 2)}"
    return hashlib.sha1(key.encode()).hexdigest()[:16]

def reconcile_state():
    """Detect drift between state.json and what's actually on disk."""
    state = load_state()
    changed = False
    for clip_id, entry in list(state.items()):
        if not Path(entry["output_path"]).exists():
            print(f"[reconcile] state points to missing file, dropping entry: {entry['output_path']}")
            del state[clip_id]
            changed = True
    known = {Path(e["output_path"]).name for e in state.values()}
    on_disk = {p.name for p in OUTPUT_DIR.glob("*.mp4")}
    orphans = on_disk - known
    if orphans:
        print(f"[reconcile] untracked files in {OUTPUT_DIR}/: {sorted(orphans)}")
    if changed:
        save_state(state)
    return state


## 5. Boundary quality check + clip extraction (ffmpeg, atomic)

`extract_clip` re-encodes (rather than stream-copies) so the cut lands
exactly on the chosen boundary instead of snapping to the nearest keyframe,
and writes to a `.part` file that's atomically renamed into place once
ffmpeg succeeds. `verify_clean_boundaries` is a post-hoc check that
confirms the *shipped* clip actually starts/ends near silence — a second,
independent sanity check on top of the boundary-snap logic in Section 2.


In [ ]:
def verify_clean_boundaries(clip_path: Path, noise_db: float = -30, edge_tol: float = 0.08):
    sil = find_silences(clip_path, noise_db=noise_db, min_dur=0.05)
    dur = float(subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", str(clip_path)],
        capture_output=True, text=True).stdout.strip())
    starts_clean = any(s <= edge_tol for s, e in sil)
    ends_clean = any(e >= dur - edge_tol for s, e in sil)
    return starts_clean, ends_clean

def extract_clip(video_path: Path, start: float, end: float, out_path: Path,
                  pad_start: float = 0.25, pad_end: float = 0.35):
    s = max(0.0, start - pad_start)
    d = (end - start) + pad_start + pad_end
    tmp_out = out_path.with_suffix(out_path.suffix + ".part")
    cmd = [
        "ffmpeg", "-y",
        "-ss", f"{s:.3f}", "-i", str(video_path), "-t", f"{d:.3f}",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "20",
        "-c:a", "aac", "-b:a", "128k",
        "-movflags", "+faststart",
        "-f", "mp4",
        str(tmp_out),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 or not tmp_out.exists():
        if tmp_out.exists():
            tmp_out.unlink()
        raise RuntimeError(f"ffmpeg failed for {out_path.name}: {result.stderr[-800:]}")
    os.replace(tmp_out, out_path)


## 6. Orchestrator

`run_agent` calls every piece above **in sequence, each step depending on
the previous one's output**: silence detection → transcription → topic
chaptering (depends on the transcript) → boundary snap (depends on both
chaptering and silence detection) → state check → ffmpeg extraction →
state write. This is also where the idempotency check happens: for each
topic, before doing any work, it computes the clip ID and checks state +
disk; only a genuinely new clip reaches the ffmpeg extraction step.


In [ ]:
def safe_filename(topic: str) -> str:
    """Convert a topic into a safe, readable filename."""
    name = topic.lower().strip()
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name[:100]

In [ ]:
def run_agent(video_path: str, topics: List[str]) -> List[Dict]:
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(video_path)

    video_hash = sha256_file(video_path)
    state = reconcile_state()

    silences = find_silences(video_path)
    print(f"[silence] {len(silences)} real silence intervals detected in source audio")

    transcript = transcribe(video_path)
    segments = transcript["segments"]

    spans = chapter_topics(segments, topics)

    results = []
    for topic in topics:
        if topic not in spans:
            print(f"[skip] no chapter found for topic: {topic!r}")
            continue
        raw_start, raw_end = spans[topic]

        # clip_id identity is tied to the RAW (pre-snap) chapter boundaries --
        # stable across runs even if snap_chapter_boundary's window is retuned.
        clip_id = make_clip_id(video_hash, topic, raw_start, raw_end)
        video_folder = OUTPUT_DIR / video_path.stem
        video_folder.mkdir(parents=True, exist_ok=True)
        filename = f"{safe_filename(topic)}_{clip_id[:8]}.mp4"
        out_path = video_folder / filename

        existing = state.get(clip_id)
        if existing and Path(existing["output_path"]).exists():
            print(f"[skip-cached] '{topic}' already generated -> {out_path.name} (no re-encode)")
            results.append(existing)
            continue

        snapped_start = max(0.0, snap_chapter_boundary(raw_start, silences))
        snapped_end = snap_chapter_boundary(raw_end, silences)

        print(f"[generate] '{topic}' -> {raw_start:.2f}s-{raw_end:.2f}s "
              f"(snapped: {snapped_start:.2f}s-{snapped_end:.2f}s) -> {out_path.name}")
        extract_clip(video_path, snapped_start, snapped_end, out_path)

        starts_clean, ends_clean = verify_clean_boundaries(out_path)
        if not (starts_clean and ends_clean):
            print(f"[warn] '{topic}': boundary check found no near-silence at "
                  f"{'start' if not starts_clean else 'end'} of {out_path.name} "
                  f"-- source audio may genuinely have no pause near this content")

        entry = {
            "clip_id": clip_id,
            "topic": topic,
            "video": str(video_path),
            "video_hash": video_hash,
            "start": raw_start,
            "end": raw_end,
            "snapped_start": snapped_start,
            "snapped_end": snapped_end,
            "output_path": str(out_path),
            "output_sha256": sha256_file(out_path),
            "boundary_check_passed": bool(starts_clean and ends_clean),
            "created_at": time.time(),
        }
        state[clip_id] = entry
        save_state(state)  # persist after every clip, not just at the end
        results.append(entry)

    return results


## 7. Demo: run it twice on the same video

`videos/demo_clip.mp4` is a real ~214s recording of a database-systems
coursework walkthrough with three natural sections: a short welcome, a long
"deliverable 1" explanation (normalization + a stored-procedure demo), and a
"deliverable 2" explanation (a BI solution). Topics below are phrased using
words that actually ground in this content — that matters for this
chaptering approach (see Section 8 for why).

> If your video file has a different name, update the path in the cell
> below before running.

Expected behaviour:
- **Run 1**: transcribes (or reuses the cached transcript), chapters the
  full transcript into 3 topic spans, snaps each to real silence, writes 3
  `.mp4` files, writes `clip_state.json`.
- **Run 2**: reuses the cached transcript, re-derives the same chapters,
  computes the same clip IDs, sees they already exist on disk, and skips
  every single one — zero new files, zero re-encodes, zero errors.


In [ ]:
topics = [
    "welcome and demonstration of the coursework",
    "deliverable 1 database normalisation and stored procedure",
    "deliverable 2 business intelligence solution using SSAS",
]

print("=" * 20, "RUN 1", "=" * 20)
run1 = run_agent("videos/demo_clip.mp4", topics)
print(f"\nRun 1 produced/confirmed {len(run1)} clips:")
for r in run1:
    dur = r["snapped_end"] - r["snapped_start"]
    print(f"  - {r['topic']!r}: {Path(r['output_path']).name}  "
          f"({r['snapped_start']:.1f}s-{r['snapped_end']:.1f}s, {dur:.1f}s long)")


In [ ]:
files_before = sorted(p.name for p in OUTPUT_DIR.glob("*.mp4"))

print("=" * 20, "RUN 2 (same input)", "=" * 20)
run2 = run_agent("videos/demo_clip.mp4", topics)

files_after = sorted(p.name for p in OUTPUT_DIR.glob("*.mp4"))

assert files_before == files_after, "Idempotency violated: file set changed between runs!"
assert [r["clip_id"] for r in run1] == [r["clip_id"] for r in run2], "Clip IDs differ between runs!"
print(f"\nOK: run 2 touched 0 new files. Clip count stable at {len(files_after)}.")
print("Files:", files_after)


## 8. Boundary quality check, and a known limitation

Two separate questions here. First: does each shipped clip actually
start/end on silence, not mid-word? Second, separately: how close did the
chaptering step get to the true topic boundaries?


In [ ]:
print("=" * 20, "BOUNDARY QUALITY", "=" * 20)
for r in run2:
    status = "OK" if r.get("boundary_check_passed") else "REVIEW"
    print(f"[{status}] '{r['topic']}' -> {Path(r['output_path']).name}  "
          f"({r['snapped_start']:.2f}s-{r['snapped_end']:.2f}s)")


**Known limitation**, found while testing on this real video: the
transcript's true deliverable-1 → deliverable-2 transition is at ~131.8s →
132.8s (there's a strong silence gap right there — confirmed independently
by `find_silences`). The chaptering step above lands the boundary at
~155–157s instead, about 24 seconds late.

The reason is instructive: the transcript says *"We will now briefly
explain **D2** which is our **BI solution** developed using **SQL server
analysis services**"* — none of "D2", "BI", or "SSAS" share a token with
the topic string `"deliverable 2 business intelligence solution using
SSAS"` written out in full. Token-overlap scoring has nothing to grab onto
right at the true boundary, so the monotonic DP keeps attributing a few
more sentences to deliverable 1 (which *do* share words like "database")
before the score finally tips over.

This is exactly the class of problem semantic matching solves and lexical
matching structurally can't: an LLM call over the full transcript would get
this right, because it can understand "D2" means "deliverable 2" and "BI
solution" means "business intelligence solution" without needing matching
vocabulary. With more time, that's the first thing I'd change — see "Next
steps" below.

**Practical takeaway for using this agent as it stands**: phrase topics
using vocabulary that actually appears in the video, and treat the
low-confidence warning it logs as a signal to rephrase the topic, not
something to ignore.


## Notes for write-up

- **How segments are chosen**: the agent first decides the full contiguous
  chapter each topic covers (monotonic-DP keyword chaptering over the
  whole transcript), *then* snaps just the two edges of that chapter to
  real silence. That keeps the "cut on a pause, not a hard timestamp"
  property while making sure each clip covers a topic's full discussion,
  not just the sentence where a keyword happened to appear.
- **Two tools in sequence, second depends on first's output**: transcription
  (STT, `faster-whisper`) produces the segment list; topic chaptering
  consumes that segment list and could not run without it; ffmpeg
  extraction then consumes the chaptering's output. `find_silences` (also
  ffmpeg) is a third, independent tool used for boundary refinement.
- **Idempotency mechanism**: content-addressable IDs
  (`sha1(video_hash:topic:start:end)`) computed from the pre-snap chapter
  boundaries, not a run counter. The second run computes the *same* ID and
  correctly treats it as the same clip.
- **Crash safety**: both `clip_state.json` and each `.mp4` are written to a
  temp path and atomically renamed into place.
- **Cost control**: transcription is cached per video hash; a re-run on an
  unchanged file never re-invokes Whisper.
- **Auditability**: every clip's state entry stores both the raw chapter
  boundaries and the snapped extraction boundaries, so a reviewer can see
  exactly how the final cut was derived.
- **Honest limitation, not hidden**: chaptering is lexical, not semantic, so
  it can mis-place a boundary when the topic label and the transcript use
  different words for the same thing (documented with a concrete example
  in Section 8). Confidence is logged per topic so a weak match isn't
  silently trusted.

## Next steps with more time

1. **Swap keyword chaptering for an LLM call**: send the indexed,
   timestamped transcript plus the topic list in one call and ask for the
   full sentence-index range each topic covers. This directly fixes the
   ~24s-late boundary documented in Section 8 and would let topic phrasing
   be arbitrary instead of needing to match transcript vocabulary. Since
   chaptering output would change, clip IDs would change too — the
   existing idempotency design already handles that correctly (different
   chaptering → different `clip_id` → new clips generated, no silent reuse
   of stale ones).
2. **Tests**: unit tests for `keyword_score`, the monotonic DP, and
   `snap_chapter_boundary` on synthetic silence lists, plus an integration
   test that runs the full agent twice on a short fixture video and asserts
   zero new files on the second run (currently this is only demonstrated
   interactively in Section 7, not asserted in CI).
3. **CLI ergonomics**: proper argument parsing for video path, topics file,
   and thresholds (noise dB, snap window) instead of editing constants in
   the script.
4. **Multiple videos / batch mode**: `run_agent` already keys everything by
   video hash, so batching over a folder of videos is mostly plumbing.
5. **Automatic topic discovery**: right now the user supplies the topic
   list. An LLM pass over the transcript could propose topics instead of
   requiring them upfront.
